# 918 Bail Bonds Advisory AI on Kaggle GPU

Synthetic, reproducible demo for organizing consented bail-workflow evidence. This is not a criminal-risk, flight-risk, detention, eligibility, pricing, or legal-decision model. A licensed bondsman remains the decision-maker.

Learning goals:
- run an explainable case-readiness assessment;
- identify missing intake fields and human-review actions;
- optionally batch public-safe evidence embeddings on a Kaggle GPU.

Use synthetic data only.

## Safety boundary

This notebook never produces a risk score or recommendation about whether someone should receive bail. It returns human_review_required, workflow completeness, public-source review counts, and explicit next actions. It must not automate a legal, liberty, insurance, credit, or payment decision.

In [ ]:
import json, importlib.util
from pprint import pprint
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU fallback"
except Exception:
    torch = None
    DEVICE = "cpu"
    GPU_NAME = "CPU fallback"
print({"device": DEVICE, "device_name": GPU_NAME, "torch_available": torch is not None})

In [ ]:
# Synthetic public-safe examples only.
examples = [
    {"case_id":"demo-001","full_name":"Alex Example","date_of_birth":"1990-01-01","phone":"918-555-0100","county":"Tulsa","consent":True,"emergency":True,"source_matches":1},
    {"case_id":"demo-002","full_name":"Jordan Example","date_of_birth":"","phone":"918-555-0101","county":"Tulsa","consent":True,"emergency":False,"source_matches":0},
]
pprint(examples)

In [ ]:
def assess_review_readiness(intake):
    required = ("full_name", "date_of_birth", "phone", "consent")
    missing = [field for field in required if not intake.get(field)]
    matches = int(intake.get("source_matches", 0))
    actions = []
    if missing:
        actions.append("collect_missing_consented_intake_fields")
    actions.append("licensed_bondsman_confirm_source_match" if matches else "licensed_bondsman_review_source_status")
    actions.append("licensed_bondsman_record_human_decision")
    return {
        "assessment_type": "non_binding_case_readiness",
        "decision": "human_review_required",
        "workflow_priority": "urgent" if intake.get("emergency") else "standard",
        "missing_information": missing,
        "evidence_summary": {"source_match_count": matches, "human_source_confirmation_required": True},
        "required_human_actions": actions,
        "explanation": "Workflow evidence only; not a risk, eligibility, detention, pricing, or legal decision.",
    }
packets = [assess_review_readiness(item) for item in examples]
for packet in packets:
    print(json.dumps(packet, indent=2))

## Optional GPU evidence organization

With Kaggle Internet enabled, the next cell loads a compact sentence-embedding model and batches public-safe evidence descriptions on the GPU. Similarity is only an organization aid for a human reviewer; it is not identity verification or a legal recommendation. The fallback keeps the notebook runnable without model downloads.

In [ ]:
# Never pass names, DOBs, phone numbers, or raw case reports here.
evidence_text = [
    "public booking record requires licensed bondsman confirmation",
    "no public source match was returned; review source status",
    "consented intake is missing a required field",
]
backend = "keyword-fallback"
embeddings = None
if importlib.util.find_spec("sentence_transformers"):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
    embeddings = model.encode(evidence_text, batch_size=32, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    backend = "sentence-transformers"
    print("embedding backend:", backend, "shape:", tuple(embeddings.shape), "device:", DEVICE)
else:
    print("sentence-transformers is not installed; using transparent keyword fallback")
print({"backend": backend, "gpu_optimized": DEVICE == "cuda" and backend == "sentence-transformers"})

In [ ]:
summary = {
    "packets_generated": len(packets),
    "all_require_human_review": all(p["decision"] == "human_review_required" for p in packets),
    "risk_score_present": any("risk_score" in p for p in packets),
    "gpu_device": GPU_NAME,
}
pprint(summary)
assert summary["all_require_human_review"]
assert not summary["risk_score_present"]

## Exercise

Add a third synthetic intake with consent missing. Verify that the assessment requests missing consent and still requires a licensed bondsman decision. Do not add a risk label or score.

In [ ]:
exercise_case = {"case_id":"demo-003","full_name":"Taylor Example","date_of_birth":"1988-02-02","phone":"918-555-0102","county":"Tulsa","consent":False,"emergency":False,"source_matches":2}
exercise_packet = assess_review_readiness(exercise_case)
pprint(exercise_packet)
assert "consent" in exercise_packet["missing_information"]
assert exercise_packet["decision"] == "human_review_required" 

## Production integration

The local app exposes the same safe contract at:
GET /api/requests/<request_id>/assessment

Private intake remains off-chain. Production use requires authentication, consent, retention controls, licensed operator review, jurisdiction-specific compliance, and independent legal/security review.